# Popularity bias in recommenders

Cleans Amazon Reviews 2023 data (`prepare.py`), splits it by time (`split.py`), then trains and evaluates
three recommenders (`mostpop`, `itemcf`, `mf`) on accuracy and popularity-bias metrics (`run.py`).

Run this notebook from the project folder. Data: https://amazon-reviews-2023.github.io/

## Settings

In [ ]:
CATEGORY = "Clothing_Shoes_and_Jewelry"
RAW_REVIEWS = f"data/{CATEGORY}.jsonl.gz"
K_CORE = 2
MODELS = ["mostpop", "itemcf", "mf"]

## Download the raw reviews
Downloads `<CATEGORY>.jsonl.gz` from the Amazon Reviews 2023 site into `data/` (Clothing is ~7 GB).
Skipped if the raw file or the cleaned parquet already exists.

In [ ]:
import os
import urllib.request

URL = f"https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/{CATEGORY}.jsonl.gz"

def progress(n, size, total):
    if n % 12_000 == 0:  # about every 100 MB
        print(f"\r{n * size / 1e9:.1f} / {total / 1e9:.1f} GB", end="")

if os.path.exists(f"data/{CATEGORY}_interactions.parquet") or os.path.exists(RAW_REVIEWS):
    print("Already have the data, skipping download")
else:
    os.makedirs("data", exist_ok=True)
    urllib.request.urlretrieve(URL, RAW_REVIEWS + ".part", progress)  # .part so a failed download isn't mistaken for a full one
    os.replace(RAW_REVIEWS + ".part", RAW_REVIEWS)
    print("\nDone")

## Clean the raw reviews
Runs `prepare.py` on the raw reviews to make `data/<CATEGORY>_interactions.parquet`. Skipped if that file already exists.

In [ ]:
import sys
PY = sys.executable  # run the scripts with this notebook's Python, not whatever "python" is on PATH

if os.path.exists(f"data/{CATEGORY}_interactions.parquet"):
    print("Cleaned parquet already exists, skipping prepare.py")
else:
    !"{PY}" prepare.py {RAW_REVIEWS} {CATEGORY}

## Split into train / validation / test

In [ ]:
!"{PY}" split.py {CATEGORY} {K_CORE}

## Train and evaluate

In [ ]:
for m in MODELS:
    !"{PY}" run.py {CATEGORY} {K_CORE} {m}